# Notebook 03_1 — Predictive Performance (Text + Image Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Men's Shoes (Size 8)

---

Same as 03_2 but uses **multimodal embeddings** (RoBERTa + BEiT + SAINT).  
Adds `x_emb` as a fourth feature specification on top of x, x_pca, x_sim.

Model specifications compared:
- OLS / Boosting — Tabular only
- OLS / Boosting — Tabular + PCA
- OLS / Boosting — Tabular + Similarities
- OLS / Boosting — Tabular + Embeddings (multimodal)
- Deep Time Independent
- Deep Time Dependent

## ① Mount Drive

In [1]:
# Drive mount not needed for local execution
print('✅ Local mode')

✅ Local mode


## ② Set Working Directory

In [2]:
import os, sys
from pathlib import Path

# Detect project root by walking up from CWD to find 'data/' and 'code/' folders
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

# Change to code/ directory so relative imports and paths work
CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files here: {os.listdir(".")}')

Working directory: /home/iankuzuma/claude_code/demand-modeling-data-men-8-whole/code
Files here: ['utils', '01_1_create_dataset_txt_img.ipynb', '01_2_create_dataset_txt.ipynb', '02_cluster_centroid_products.ipynb', '02_cluster_centroid_products_random20.ipynb', '03_1_predictive_performance_txt_img.ipynb', '03_2_predictive_performance_txt.ipynb', '04_evaluation.ipynb', 'requirements.txt', 'main_train_keys.csv', 'main_val_keys.csv']


## ③ Imports

In [3]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)
print('✅ Imports done')

✅ Imports done


## ④ Load Dataset

Loads the **multimodal** embeddings dataset from notebook 01_1 (txt_only=False).
Key difference from 03_2: this dataset includes both text AND image embeddings.

In [4]:
txt_only = False
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"

df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val   = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val   = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val   = df_full_val.dropna()

dummy_subcat_names = [category for category in df_full_val["subcat_aggregated"].unique()]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Train shape: {df_full_train.shape}")
print(f"Val shape:   {df_full_val.shape}")
print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Train shape: (12348, 332)
Val shape:   (12384, 332)
Dummy subcat names: ['Loafers & Slip-Ons', 'Other', 'Road Running', 'Oxfords', 'Shoes', 'Fashion Sneakers', 'Water Shoes', 'Walking', 'Trail Running', 'Golf', 'Track & Field & Cross Country']
All time steps: ['2025-04-28', '2025-05-26', '2025-06-23', '2025-07-21', '2025-08-18', '2025-09-15', '2025-10-13', '2025-11-10', '2025-12-08', '2026-01-05', '2026-02-02', '2026-03-02']


Index(['ASIN', 'date', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=332)

## ⑤ Sanity Check — Row Counts

In [5]:
print(f"Val rows:   {len(df_full_val)}")
print(f"Train rows: {len(df_full_train)}")

Val rows:   12384
Train rows: 12348


## ⑥ Define Controls and Feature Specifications

Same controls as 03_2 plus `controls_emb` — all 256 multimodal embedding dimensions.

In [6]:
n_lags = 1

outcome   = "Q_t"
treatment = "P_bb_t"

outcome_diff   = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls
additional_controls_deep = [var for var in additional_controls if var not in dummy_subcat_names]

# Key difference from 03_2 — multimodal emb columns
controls_emb = [var for var in df_full_train.columns if "emb" in var]

print(f"Continuous controls:    {len(cont_controls)}")
print(f"Dummy controls:         {len(dummy_controls)}")
print(f"Total controls:         {len(additional_controls)}")
print(f"Embedding columns:      {len(controls_emb)}")

Continuous controls:    5
Dummy controls:         23
Total controls:         28
Embedding columns:      256


## ⑦ Initialize Results DataFrames

In [7]:
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df      = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)
print('✅ Results DataFrames initialized')

✅ Results DataFrames initialized


## ⑧ Deep Model R² — Level

In [8]:
df_dict = {"Train": df_full_train, "Test": df_full_val}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome (time independent):   {r2_ml_l}")
    print(f"  R2 Treatment (time independent):  {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"  R2 Outcome (lag1):                {r2_ml_l_lag1}")
    print(f"  R2 Treatment (lag1):              {r2_ml_m_lag1}")
    print()

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
  R2 Outcome (time independent):   0.8093
  R2 Treatment (time independent):  0.7849
  R2 Outcome (lag1):                0.8868
  R2 Treatment (lag1):              0.7845

Evaluation for Test set
  R2 Outcome (time independent):   0.6517
  R2 Treatment (time independent):  0.7458
  R2 Outcome (lag1):                0.8259
  R2 Treatment (lag1):              0.7508



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.8093,0.6517,0.7849,0.7458
Deep Time Dependent,0.8868,0.8259,0.7845,0.7508


## ⑨ Deep Model R² — Diff

In [9]:
results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome diff (time independent):   {r2_ml_l_diff}")
    print(f"  R2 Treatment diff (time independent):  {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"  R2 Outcome diff (lag1):                {r2_ml_l_diff_lag1}")
    print(f"  R2 Treatment diff (lag1):              {r2_ml_m_diff_lag1}")
    print()

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
  R2 Outcome diff (time independent):   0.06702063
  R2 Treatment diff (time independent):  -0.00075128
  R2 Outcome diff (lag1):                0.13560806
  R2 Treatment diff (lag1):              -0.00472447

Evaluation for Test set
  R2 Outcome diff (time independent):   0.03822997
  R2 Treatment diff (time independent):  -0.00434512
  R2 Outcome diff (lag1):                0.09990855
  R2 Treatment diff (lag1):              -0.00873628



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.067021,0.038230,-0.000751,-0.004345
Deep Time Dependent,0.135608,0.099909,-0.004724,-0.008736


## ⑩ Build Feature Matrices

Key difference from 03_2: adds `x_train_emb` and `x_test_emb`
using all 256 multimodal embedding dimensions.

In [10]:
# Train set
y_train      = df_full_train[outcome].squeeze()
d_train      = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train     = sm.add_constant(df_full_train[additional_controls])
x_train_pca = sm.add_constant(df_full_train[additional_controls + controls_pca])
x_train_sim = sm.add_constant(df_full_train[additional_controls + controls_similarities])
x_train_emb = sm.add_constant(df_full_train[additional_controls + controls_emb])

# Test set
y_test      = df_full_val[outcome].squeeze()
d_test      = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test     = sm.add_constant(df_full_val[additional_controls])
x_test_pca = sm.add_constant(df_full_val[additional_controls + controls_pca])
x_test_sim = sm.add_constant(df_full_val[additional_controls + controls_similarities])
x_test_emb = sm.add_constant(df_full_val[additional_controls + controls_emb])

# Rename columns for LightGBM
for df in [x_train, x_test, x_train_pca, x_test_pca,
           x_train_sim, x_test_sim, x_train_emb, x_test_emb]:
    df.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x), inplace=True)

print(f"x_train shape:     {x_train.shape}")
print(f"x_train_pca shape: {x_train_pca.shape}")
print(f"x_train_sim shape: {x_train_sim.shape}")
print(f"x_train_emb shape: {x_train_emb.shape}")

x_train shape:     (12348, 29)
x_train_pca shape: (12348, 34)
x_train_sim shape: (12348, 34)
x_train_emb shape: (12348, 285)


## ⑪ Build Dict Structures

In [11]:
train_dict = {
    "y": y_train, "y_diff": y_train_diff,
    "d": d_train, "d_diff": d_train_diff,
    "x": x_train, "x_pca": x_train_pca,
    "x_sim": x_train_sim, "x_emb": x_train_emb,
}

test_dict = {
    "y": y_test, "y_diff": y_test_diff,
    "d": d_test, "d_diff": d_test_diff,
    "x": x_test, "x_pca": x_test_pca,
    "x_sim": x_test_sim, "x_emb": x_test_emb,
}
print('✅ Train and test dicts ready')

✅ Train and test dicts ready


## ⑫ Tabular Models — Level

Four feature specifications including the new x_emb (Tabular + full embeddings).

In [12]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y"])
    print(f"  R2 Outcome train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d"])
    print(f"  R2 Treatment train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_tab = pd.concat([results_df_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_tab.columns = column_names
results_df_tab.index   = row_names
results_df_tab


 Feature specification: x
  OLS
  R2 Outcome train/test: 0.2612 / 0.3068
  R2 Treatment train/test: 0.3081 / 0.3011
  Boosting


  R2 Outcome train/test: 0.7053 / 0.5852


  R2 Treatment train/test: 0.7008 / 0.4942

 Feature specification: x_pca
  OLS
  R2 Outcome train/test: 0.6816 / 0.5595
  R2 Treatment train/test: 0.7074 / 0.6623
  Boosting


  R2 Outcome train/test: 0.9166 / 0.7027


  R2 Treatment train/test: 0.9106 / 0.7481

 Feature specification: x_sim
  OLS
  R2 Outcome train/test: 0.6923 / 0.5630
  R2 Treatment train/test: 0.7038 / 0.6683
  Boosting


...

  R2 Treatment train/test: 0.8879 / 0.7335

 Feature specification: x_emb
  OLS
  R2 Outcome train/test: 0.8239 / 0.5819


  R2 Treatment train/test: 0.8330 / 0.7076
  Boosting


  R2 Outcome train/test: 0.9394 / 0.7063


  R2 Treatment train/test: 0.9453 / 0.7675


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.261236,0.306847,0.308105,0.301104
Boosting (Tabular),0.705278,0.585167,0.700764,0.494189
OLS (Tabular + PCA),0.681607,0.559499,0.707424,0.662308
Boosting (Tabular + PCA),0.916582,0.702748,0.910567,0.748063
OLS (Tabular + Similarities),0.692321,0.563020,0.703842,0.668304
Boosting (Tabular + Similarities),0.896677,0.690325,0.887872,0.733476
OLS (Tabular + Embeddings),0.823903,0.581866,0.833009,0.707594
Boosting (Tabular + Embeddings),0.939419,0.706322,0.945299,0.767498


## ⑬ Summary — Level Models

In [13]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.261236,0.306847,0.308105,0.301104
Boosting (Tabular),0.705278,0.585167,0.700764,0.494189
OLS (Tabular + PCA),0.681607,0.559499,0.707424,0.662308
Boosting (Tabular + PCA),0.916582,0.702748,0.910567,0.748063
OLS (Tabular + Similarities),0.692321,0.563020,0.703842,0.668304
Boosting (Tabular + Similarities),0.896677,0.690325,0.887872,0.733476
OLS (Tabular + Embeddings),0.823903,0.581866,0.833009,0.707594
Boosting (Tabular + Embeddings),0.939419,0.706322,0.945299,0.767498
Deep Time Independent,0.809300,0.651700,0.784900,0.745800
Deep Time Dependent,0.886800,0.825900,0.784500,0.750800


## ⑭ Tabular Models — Diff

In [14]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y_diff"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome diff train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d_diff"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d_diff"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment diff train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y_diff"])
    print(f"  R2 Outcome diff train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d_diff"])
    print(f"  R2 Treatment diff train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_diff_tab = pd.concat([results_df_diff_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_diff_tab.columns = column_names
results_df_diff_tab.index   = row_names
results_df_diff_tab


 Feature specification: x
  OLS
  R2 Outcome diff train/test: 0.0773 / 0.0621
  R2 Treatment diff train/test: 0.0155 / 0.0134
  Boosting


  R2 Outcome diff train/test: 0.3438 / 0.1658


  R2 Treatment diff train/test: 0.1456 / 0.0105

 Feature specification: x_pca
  OLS
  R2 Outcome diff train/test: 0.0797 / 0.0636
  R2 Treatment diff train/test: 0.0159 / 0.0138
  Boosting


  R2 Outcome diff train/test: 0.4753 / 0.2140


  R2 Treatment diff train/test: 0.2322 / -0.0014

 Feature specification: x_sim
  OLS
  R2 Outcome diff train/test: 0.0796 / 0.0639
  R2 Treatment diff train/test: 0.0159 / 0.0140
  Boosting


...

  R2 Treatment diff train/test: 0.2098 / 0.0013

 Feature specification: x_emb
  OLS
  R2 Outcome diff train/test: 0.1122 / 0.0243


  R2 Treatment diff train/test: 0.0356 / -0.0262
  Boosting


  R2 Outcome diff train/test: 0.5769 / 0.1880


  R2 Treatment diff train/test: 0.3546 / -0.0253


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.077345,0.062065,0.015537,0.013373
Boosting (Tabular),0.343770,0.165774,0.145633,0.010545
OLS (Tabular + PCA),0.079722,0.063645,0.015931,0.013799
Boosting (Tabular + PCA),0.475296,0.214025,0.232223,-0.001398
OLS (Tabular + Similarities),0.079646,0.063884,0.015852,0.013955
Boosting (Tabular + Similarities),0.421616,0.170805,0.209769,0.001345
OLS (Tabular + Embeddings),0.112169,0.024346,0.035648,-0.026228
Boosting (Tabular + Embeddings),0.576884,0.188025,0.354552,-0.025318


## ⑮ Summary — Diff Models

In [15]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.077345,0.062065,0.015537,0.013373
Boosting (Tabular),0.343770,0.165774,0.145633,0.010545
OLS (Tabular + PCA),0.079722,0.063645,0.015931,0.013799
Boosting (Tabular + PCA),0.475296,0.214025,0.232223,-0.001398
OLS (Tabular + Similarities),0.079646,0.063884,0.015852,0.013955
Boosting (Tabular + Similarities),0.421616,0.170805,0.209769,0.001345
OLS (Tabular + Embeddings),0.112169,0.024346,0.035648,-0.026228
Boosting (Tabular + Embeddings),0.576884,0.188025,0.354552,-0.025318
Deep Time Independent,0.067021,0.038230,-0.000751,-0.004345
Deep Time Dependent,0.135608,0.099909,-0.004724,-0.008736


## ⑯ Final Summary (% format)

R² multiplied by 100 — matches paper Table 2 format.
Compare with 03_2 (txt only) to see the gain from adding image embeddings.

In [16]:
print("=== Level Models — Test R² (%) ===")
print(results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100)
print()
print("=== Diff Models — Test R² (%) ===")
print(results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100)

=== Level Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          30.68      30.11
Boosting (Tabular)                     58.52      49.42
OLS (Tabular + PCA)                    55.95      66.23
Boosting (Tabular + PCA)               70.27      74.81
OLS (Tabular + Similarities)           56.30      66.83
Boosting (Tabular + Similarities)      69.03      73.35
OLS (Tabular + Embeddings)             58.19      70.76
Boosting (Tabular + Embeddings)        70.63      76.75
Deep Time Independent                  65.17      74.58
Deep Time Dependent                    82.59      75.08

=== Diff Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                           6.21       1.34
Boosting (Tabular)                     16.58       1.05
OLS (Tabular + PCA)                     6.36       1.38
Boosting (Tabular + PCA)               21.40      -0.14
OLS (Tabular + Similarities)      